In [1]:
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files

model_name = "google/gemma-2-2b"
transcoder_name = "mntss/clt-gemma-2-2b-2.5M"
backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
model = ReplacementModel.from_pretrained(
    model_name, transcoder_name, dtype=torch.bfloat16, lazy_encoder=True, backend=backend
)

Fetching 52 files:   0%|          | 0/52 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


In [2]:
prompt = "Fact: The capital of state containing Dallas is"  # What you want to get the graph for
max_n_logits = 10  # How many logits to attribute from, max. We attribute to min(max_n_logits, n_logits_to_reach_desired_log_prob); see below for the latter
desired_logit_prob = 0.95  # Attribution will attribute from the minimum number of logits needed to reach this probability mass (or max_n_logits, whichever is lower)
max_feature_nodes = 8192  # Only attribute from this number of feature nodes, max. Lower is faster, but you will lose more of the graph. None means no limit.
batch_size = 256  # Batch size when attributing
offload = (
    "cpu"
)  # Offload various parts of the model during attribution to save memory. Can be 'disk', 'cpu', or None (keep on GPU)
verbose = True  # Whether to display a tqdm progress bar and timing report

In [3]:
graph = attribute(
    prompt=prompt,
    model=model,
    max_n_logits=max_n_logits,
    desired_logit_prob=desired_logit_prob,
    batch_size=batch_size,
    max_feature_nodes=max_feature_nodes,
    offload=offload,
    verbose=verbose,
)

Phase 0: Precomputing activations and vectors
Precomputation completed in 7.32s
Found 2932 active features
Phase 1: Running forward pass
Forward pass completed in 0.08s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.7031
Will include 2932 of 2932 feature nodes
Input vectors built in 1.82s
Phase 3: Computing logit attributions
<sys>:0: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
Logit attributions completed in 0.21s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 2932/2932 [00:00<00:00, 4718.44it/s] 
Feature attributions completed in 0.62s
Attribution completed in 12.49s


In [4]:
graph_dir = "graphs"
graph_name = "example_graph.pt"
graph_dir = Path(graph_dir)
graph_dir.mkdir(exist_ok=True)
graph_path = graph_dir / graph_name

graph.to_pt(graph_path)

In [ ]:
from circuit_tracer.graph import Graph
from summarization.prune import prune_pt_graph
  
graph = Graph.from_pt("graphs/example_graph.pt")
pg = prune_pt_graph(graph, node_threshold=0.01, edge_threshold=0.98)

In [10]:
pg.num_nodes, pg.num_edges

(20, 100)

In [11]:
pg

PruneGraph(nodes=[Node(node_id='E_2_0', node_idx=0, feature=0, layer='E', ctx_idx=0, feature_type='embedding', token_prob=0.0, is_target_logit=False, run_idx=0, reverse_ctx_idx=0, jsNodeId='E_2-0', clerp='', influence=None, activation=None, relevance=None), Node(node_id='E_18143_1', node_idx=1, feature=1, layer='E', ctx_idx=1, feature_type='embedding', token_prob=0.0, is_target_logit=False, run_idx=0, reverse_ctx_idx=0, jsNodeId='E_18143-1', clerp='', influence=None, activation=None, relevance=None), Node(node_id='E_235292_2', node_idx=2, feature=2, layer='E', ctx_idx=2, feature_type='embedding', token_prob=0.0, is_target_logit=False, run_idx=0, reverse_ctx_idx=0, jsNodeId='E_235292-2', clerp='', influence=None, activation=None, relevance=None), Node(node_id='E_714_3', node_idx=3, feature=3, layer='E', ctx_idx=3, feature_type='embedding', token_prob=0.0, is_target_logit=False, run_idx=0, reverse_ctx_idx=0, jsNodeId='E_714-3', clerp='', influence=None, activation=None, relevance=None), 

In [3]:
from circuit_tracer.utils.create_graph_files import create_graph_files_from_prune_graph

create_graph_files_from_prune_graph(pg, slug="example", output_path="graphs/example_graph.json")

In [7]:
from circuit_tracer.frontend.local_server import serve


port = 8046
server = serve(data_dir="graphs/", port=port)

from IPython.display import IFrame

print(f"Use the IFrame below, or open your graph here: f'http://localhost:{port}/index.html'")
display(IFrame(src=f"http://localhost:{port}/index.html", width="100%", height="800px"))

Use the IFrame below, or open your graph here: f'http://localhost:8046/index.html'


In [6]:
server.stop()